# How to Evaluate Shape Correspondences

This notebook demonstrates how to use the evaluation metrics in `geomfum.eval` to assess the quality of shape correspondences.

## Setup

First, let's load two shapes and compute a correspondence using the Matcher.

In [14]:
import gsops.backend as gs

from geomfum.dataset import NotebooksDataset
from geomfum.matcher import FeatureMatcher, FunctionalMapMatcher
from geomfum.shape import TriangleMesh

dataset = NotebooksDataset()
mesh_a = TriangleMesh.from_file(dataset.get_filename("faust-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("faust-04"))


INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-00.off').
INFO:root:Data has already been downloaded... using cached file ('C:\Users\giuli\.geomfum\data\faust-04.off').


In [15]:
# Compute correspondence using the matcher
# matcher(shape_a, shape_b) returns fmap12 (A->B) and p2p21 (B->A)
matcher = FeatureMatcher()
result = matcher(mesh_a, mesh_b)

p2p21 = result.p2p21
print(f"P2P21 shape: {p2p21.shape}")  # Maps each vertex in B to a vertex in A

P2P21 shape: (6890,)


## Evaluation Metrics

The `geomfum.eval` module provides several metrics to evaluate correspondences:

1. **Normalized Geodesic Error** - Mean geodesic distance between predicted and ground truth correspondences
2. **Normalized Euclidean Error** - Mean Euclidean distance between predicted and ground truth correspondences  
3. **Dirichlet Energy** - Measures smoothness of the mapping
4. **Coverage** - Area-weighted fraction of target shape covered by the mapping

In [16]:
from geomfum.eval import (
    coverage,
    coverage_count,
    dirichlet_energy,
    evaluate_correspondence,
    normalized_euclidean_error,
    normalized_geodesic_error,
)

## Metrics Without Ground Truth

Some metrics can be computed without ground truth correspondences:

### Dirichlet Energy

The Dirichlet energy measures the smoothness of the mapping. Lower values indicate a smoother, more continuous correspondence.

In [17]:
energy = dirichlet_energy(mesh_a, mesh_b, p2p21)
print(f"Dirichlet Energy: {energy:.4f}")

Dirichlet Energy: 0.0082


### Coverage

Coverage measures what fraction of the target shape is mapped to. A value of 1.0 means every target vertex is reached.

In [18]:
# Area-weighted coverage
cov = coverage(mesh_a, mesh_b, p2p21)
print(f"Area-weighted Coverage: {cov:.4f}")

# Count-based coverage (simpler metric)
cov_count = coverage_count(mesh_a, mesh_b, p2p21)
print(f"Count-based Coverage: {cov_count:.4f}")

Area-weighted Coverage: 0.5828
Count-based Coverage: 0.5057


## Metrics With Ground Truth

When ground truth correspondences are available, we can compute error metrics.

In [19]:
# For shapes from the same category (e.g., cat-00 and cat-01),
# we often have identity correspondence as ground truth
corr_a = gs.arange(mesh_a.n_vertices)
corr_b = gs.arange(mesh_b.n_vertices)

### Normalized Geodesic Error

This is the most common metric for evaluating shape correspondences. It measures the mean geodesic distance between predicted and ground truth correspondences, normalized by the shape's area.

In [20]:
geo_error = normalized_geodesic_error(mesh_a, mesh_b, p2p21, corr_a, corr_b)
print(f"Normaldized Geodesic Error: {geo_error:.4f}")

Normaldized Geodesic Error: 0.0508


### Normalized Euclidean Error

Similar to geodesic error but uses Euclidean distances. Faster to compute but less accurate for curved surfaces.

In [21]:
euc_error = normalized_euclidean_error(mesh_a, mesh_b, p2p21, corr_a, corr_b)
print(f"Normalized Euclidean Error: {euc_error:.4f}")

Normalized Euclidean Error: 0.0322


## All Metrics at Once

Use `evaluate_correspondence` to compute all available metrics in one call:

In [22]:
metrics = evaluate_correspondence(mesh_a, mesh_b, p2p21, corr_a, corr_b)

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

geodesic_error: 0.0508
euclidean_error: 0.0322
dirichlet_energy: 0.0082
coverage: 0.5828
coverage_count: 0.5057


## Comparing Different Methods

Let's compare the quality of the correspondence before and after refinement:

In [23]:
from geomfum.matcher import MatcherConfig

# Quick matcher (no refinement)
config_no_refine = MatcherConfig(refiners=[])
matcher_no_refine = FunctionalMapMatcher(config=config_no_refine)
result_no_refine = matcher_no_refine(mesh_a, mesh_b)

# Default matcher (with refinement)
result_refined = result  # Already computed above

In [24]:
# Compare metrics
print("Without refinement:")
metrics_no_refine = evaluate_correspondence(
    mesh_a, mesh_b, result_no_refine.p2p21, corr_a, corr_b
)
for name, value in metrics_no_refine.items():
    print(f"  {name}: {value:.4f}")

print("\nWith refinement:")
metrics_refined = evaluate_correspondence(
    mesh_a, mesh_b, result_refined.p2p21, corr_a, corr_b
)
for name, value in metrics_refined.items():
    print(f"  {name}: {value:.4f}")

Without refinement:
  geodesic_error: 0.0250
  euclidean_error: 0.0235
  dirichlet_energy: 0.0014
  coverage: 0.5851
  coverage_count: 0.4443

With refinement:
  geodesic_error: 0.0508
  euclidean_error: 0.0322
  dirichlet_energy: 0.0082
  coverage: 0.5828
  coverage_count: 0.5057
